In [1]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/susmitmahato/codess/core/kitnet.py
/kaggle/input/datasets/susmitmahato/codess/core/_scipy_shim.py
/kaggle/input/datasets/susmitmahato/codess/core/feature_mapper.py
/kaggle/input/datasets/susmitmahato/codess/core/kitsune.py
/kaggle/input/datasets/susmitmahato/codess/core/feature_extractor.py
/kaggle/input/datasets/susmitmahato/codess/core/inc_stat.py
/kaggle/input/datasets/susmitmahato/codess/phase2/reader.py
/kaggle/input/datasets/susmitmahato/codess/phase2/find_attack_ips.py
/kaggle/input/datasets/susmitmahato/codess/phase2/README.md
/kaggle/input/datasets/susmitmahato/codess/phase2/relabel_features.py
/kaggle/input/datasets/susmitmahato/codess/phase2/combine_csvs.py
/kaggle/input/datasets/susmitmahato/codess/phase2/find_attack_pcaps.py
/kaggle/input/datasets/susmitmahato/codess/phase2/pipeline.py
/kaggle/input/datasets/susmitmahato/codess/phase2/__init__.py
/kaggle/input/datasets/susmitmahato/codess/phase2/csv_baseline/run_csv_baseline.py
/kaggle/input/datasets

In [2]:
import sys

CODE_ROOT    = '/kaggle/input/datasets/susmitmahato/codess'
COMBINED_DIR = '/kaggle/input/datasets/susmitmahato/combined'
OUTPUT_DIR   = '/kaggle/working/results_phase2/full'

sys.path.insert(0, CODE_ROOT)

In [3]:
import shutil, pathlib

# Copy pipeline to writable location and patch it
src = pathlib.Path(CODE_ROOT) / 'phase2/pipeline.py'
dst = pathlib.Path('/kaggle/working/pipeline_patched.py')
shutil.copy(src, dst)

txt = dst.read_text()

# Replace slow loop with fast numpy version
old = """    feat_matrix = df[feat_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)
    feat_matrix = feat_matrix.replace([np.inf, -np.inf], 0.0)

    for idx in range(len(df)):
        feat_vec    = feat_matrix.iloc[idx].values.astype(np.float32)
        bin_label   = int(df.iloc[idx]["binary_label"])
        attack_type = str(df.iloc[idx]["attack_type"])"""

new = """    feat_matrix = df[feat_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)
    feat_matrix = feat_matrix.replace([np.inf, -np.inf], 0.0)
    feat_array  = feat_matrix.values.astype(np.float32)
    label_array = df["binary_label"].values.astype(int)
    atype_array = df["attack_type"].values
    del feat_matrix

    for idx in range(len(df)):
        feat_vec    = feat_array[idx]
        bin_label   = int(label_array[idx])
        attack_type = str(atype_array[idx])"""

dst.write_text(txt.replace(old, new))
print("Patched!" if old in txt else "WARNING: pattern not found — check pipeline.py version")

Patched!


In [4]:
import importlib, sys
from pathlib import Path

# Load patched pipeline
spec = importlib.util.spec_from_file_location(
    "phase2.pipeline", "/kaggle/working/pipeline_patched.py"
)
mod = importlib.util.module_from_spec(spec)
sys.modules["phase2.pipeline"] = mod
spec.loader.exec_module(mod)

run_experiment = mod.run_experiment
print("Imported run_experiment successfully")

Imported run_experiment successfully


In [5]:
run_experiment(
    data_dir       = Path(COMBINED_DIR),
    output_dir     = Path(OUTPUT_DIR) / 'traditional',
    csv_suffix     = 'kitsune_115',
    feature_source = 'kitsune_network_115',
)

INFO Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO Using categorical un

[{'dataset': 'friday',
  'n_total': 1109491,
  'n_benign': 1050784,
  'n_malicious': 58707,
  'attack_rate': 0.052913,
  'AUC': 0.525194,
  'AUPRC': 0.05406,
  'EER': 0.478648,
  'TPR_at_FPR_0': 0.0,
  'FNR_at_FPR_0': 1.0,
  'threshold_FPR_0': inf,
  'TPR_at_FPR_0001': 0.0,
  'FNR_at_FPR_0001': 1.0,
  'threshold_FPR_0001': inf,
  'threshold_opt': 0.084426,
  'F1_optimal': 0.106061,
  'Precision_opt': 0.062869,
  'Recall_opt': 0.338869,
  'TP': 19894,
  'FP': 296542,
  'FN': 38813,
  'TN': 754242,
  'mean_score_benign': 0.390375,
  'mean_score_attack': 0.230376,
  'median_score_benign': 0.043357,
  'median_score_attack': 0.0495,
  'std_score_benign': 1.29812,
  'std_score_attack': 0.54895,
  'max_score': 22.387335,
  'min_score': 0.010001,
  'runtime_sec': 785.5532,
  'rows_per_sec': 1412.3691,
  'feature_source': 'kitsune_network_115',
  'n_features': 115,
  'total_rows': 1134491,
  'fm_grace': 5000,
  'ad_grace': 20000,
  'warmup_rows': 25000,
  'max_cluster_size': 10,
  'n_clusters':

In [6]:
run_experiment(
    data_dir       = Path(COMBINED_DIR),
    output_dir     = Path(OUTPUT_DIR) / 'http_only',
    csv_suffix     = 'http_85',
    feature_source = 'http_afterimage_85',
)

INFO Experiment 'http_afterimage_85' — 2 day(s) to process
INFO ======================================================================
INFO Experiment: http_afterimage_85  |  Day: friday
INFO feature_width=85  total_rows=1134491
INFO benign_prefix=138543  fm_grace=5000  ad_grace=20000
INFO Fitting FeatureMapper on 5000 rows…
INFO FeatureMapper ready — k=9 clusters
INFO Row 50000 / 1134491
INFO Row 100000 / 1134491
INFO Row 150000 / 1134491
INFO Row 200000 / 1134491
INFO Row 250000 / 1134491
INFO Row 300000 / 1134491
INFO Row 350000 / 1134491
INFO Row 400000 / 1134491
INFO Row 450000 / 1134491
INFO Row 500000 / 1134491
INFO Row 550000 / 1134491
INFO Row 600000 / 1134491
INFO Row 650000 / 1134491
INFO Row 700000 / 1134491
INFO Row 750000 / 1134491
INFO Row 800000 / 1134491
INFO Row 850000 / 1134491
INFO Row 900000 / 1134491
INFO Row 950000 / 1134491
INFO Row 1000000 / 1134491
INFO Row 1050000 / 1134491
INFO Row 1100000 / 1134491
INFO ------------------------------------------------------

[{'dataset': 'friday',
  'n_total': 1109491,
  'n_benign': 1050784,
  'n_malicious': 58707,
  'attack_rate': 0.052913,
  'AUC': 0.518869,
  'AUPRC': 0.05576,
  'EER': 0.48183,
  'TPR_at_FPR_0': 0.0,
  'FNR_at_FPR_0': 1.0,
  'threshold_FPR_0': inf,
  'TPR_at_FPR_0001': 0.001874,
  'FNR_at_FPR_0001': 0.998126,
  'threshold_FPR_0001': 3.227425,
  'threshold_opt': 0.019575,
  'F1_optimal': 0.103451,
  'Precision_opt': 0.059083,
  'Recall_opt': 0.415402,
  'TP': 24387,
  'FP': 388374,
  'FN': 34320,
  'TN': 662410,
  'mean_score_benign': 0.047212,
  'mean_score_attack': 0.052612,
  'median_score_benign': 0.015312,
  'median_score_attack': 0.01569,
  'std_score_benign': 0.215118,
  'std_score_attack': 0.250791,
  'max_score': 13.78853,
  'min_score': 0.009236,
  'runtime_sec': 463.5728,
  'rows_per_sec': 2393.3479,
  'feature_source': 'http_afterimage_85',
  'n_features': 85,
  'total_rows': 1134491,
  'fm_grace': 5000,
  'ad_grace': 20000,
  'warmup_rows': 25000,
  'max_cluster_size': 10,
 

In [7]:
run_experiment(
    data_dir       = Path(COMBINED_DIR),
    output_dir     = Path(OUTPUT_DIR) / 'hybrid',
    csv_suffix     = 'hybrid_200',
    feature_source = 'hybrid_network_http_200',
)

INFO Experiment 'hybrid_network_http_200' — 2 day(s) to process
INFO ======================================================================
INFO Experiment: hybrid_network_http_200  |  Day: friday
INFO feature_width=200  total_rows=1134491
INFO benign_prefix=138543  fm_grace=5000  ad_grace=20000
INFO Fitting FeatureMapper on 5000 rows…
INFO FeatureMapper ready — k=20 clusters
INFO Row 50000 / 1134491
INFO Row 100000 / 1134491
INFO Row 150000 / 1134491
INFO Row 200000 / 1134491
INFO Row 250000 / 1134491
INFO Row 300000 / 1134491
INFO Row 350000 / 1134491
INFO Row 400000 / 1134491
INFO Row 450000 / 1134491
INFO Row 500000 / 1134491
INFO Row 550000 / 1134491
INFO Row 600000 / 1134491
INFO Row 650000 / 1134491
INFO Row 700000 / 1134491
INFO Row 750000 / 1134491
INFO Row 800000 / 1134491
INFO Row 850000 / 1134491
INFO Row 900000 / 1134491
INFO Row 950000 / 1134491
INFO Row 1000000 / 1134491
INFO Row 1050000 / 1134491
INFO Row 1100000 / 1134491
INFO ------------------------------------------

[{'dataset': 'friday',
  'n_total': 1109491,
  'n_benign': 1050784,
  'n_malicious': 58707,
  'attack_rate': 0.052913,
  'AUC': 0.516864,
  'AUPRC': 0.052837,
  'EER': 0.483636,
  'TPR_at_FPR_0': 0.0,
  'FNR_at_FPR_0': 1.0,
  'threshold_FPR_0': inf,
  'TPR_at_FPR_0001': 0.0,
  'FNR_at_FPR_0001': 1.0,
  'threshold_FPR_0001': inf,
  'threshold_opt': 0.090767,
  'F1_optimal': 0.103115,
  'Precision_opt': 0.058839,
  'Recall_opt': 0.416611,
  'TP': 24458,
  'FP': 391216,
  'FN': 34249,
  'TN': 659568,
  'mean_score_benign': 0.402511,
  'mean_score_attack': 0.226776,
  'median_score_benign': 0.065224,
  'median_score_attack': 0.071933,
  'std_score_benign': 1.377297,
  'std_score_attack': 0.577017,
  'max_score': 22.305397,
  'min_score': 0.005558,
  'runtime_sec': 961.6425,
  'rows_per_sec': 1153.7458,
  'feature_source': 'hybrid_network_http_200',
  'n_features': 200,
  'total_rows': 1134491,
  'fm_grace': 5000,
  'ad_grace': 20000,
  'warmup_rows': 25000,
  'max_cluster_size': 10,
  'n_c

In [8]:
import json, pandas as pd

rows = []
for exp in ['traditional', 'http_only', 'hybrid']:
    for day in ['friday', 'thursday']:
        f = Path(OUTPUT_DIR) / exp / day / 'metrics.json'
        if f.exists():
            m = json.load(open(f))
            rows.append({'experiment': exp, 'day': day,
                         'AUC': m['AUC'], 'EER': m['EER'],
                         'n_attacks': m['n_malicious']})

pd.DataFrame(rows)

,experiment,day,AUC,EER,n_attacks
0,traditional,friday,0.525194,0.478648,58707
1,traditional,thursday,0.522342,0.477799,42836
2,http_only,friday,0.518869,0.481830,58707
3,http_only,thursday,0.539029,0.468001,42836
4,hybrid,friday,0.516864,0.483636,58707
5,hybrid,thursday,0.521132,0.477389,42836


In [9]:


import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RESULTS = Path("/kaggle/working/results_phase2/full")
PLOTS   = Path("/kaggle/working/extra_graphs")
PLOTS.mkdir(parents=True, exist_ok=True)

METHODS = {
    "Traditional 115": "traditional",
    "HTTP-only 85":    "http_only",
    "Hybrid 200":      "hybrid",
}
DAYS    = ["friday", "thursday"]
DAY_CAP = {"friday": "Friday", "thursday": "Thursday"}

METHOD_COLORS = {
    "Traditional 115": "#2196F3",
    "HTTP-only 85":    "#E91E63",
    "Hybrid 200":      "#4CAF50",
}
LABEL_COLORS = {0: "#2196F3", 1: "#E53935"}

def load_scores(method_key: str, day: str, sample: int = None) -> pd.DataFrame:
    path = RESULTS / method_key / day / "scores.csv"
    df = pd.read_csv(path)
    if sample and len(df) > sample:
        df = df.sample(n=sample, random_state=42).reset_index(drop=True)
    return df

def load_metrics(method_key: str, day: str) -> dict:
    path = RESULTS / method_key / day / "metrics.json"
    with open(path) as f:
        return json.load(f)

def load_curve(fname: str, method_key: str, day: str, step: int = 500) -> pd.DataFrame:
    path = RESULTS / method_key / day / fname
    df = pd.read_csv(path)
    return df.iloc[::step].reset_index(drop=True)

def load_attack_breakdown(method_key: str) -> pd.DataFrame:
    path = RESULTS / method_key / "attack_breakdown_combined.csv"
    return pd.read_csv(path)

def gaussian_kde_numpy(data: np.ndarray, x_grid: np.ndarray,
                       bw_factor: float = 1.0) -> np.ndarray:
    """Simple Gaussian KDE using numpy (no scipy required)."""
    n = len(data)
    bw = bw_factor * 1.06 * np.std(data) * n ** (-1 / 5)
    diff = x_grid[:, None] - data[None, :]           # (grid, n)
    kernel = np.exp(-0.5 * (diff / bw) ** 2) / (bw * np.sqrt(2 * np.pi))
    return kernel.mean(axis=1)

def rolling_mean_np(arr: np.ndarray, window: int) -> np.ndarray:
    """Fast rolling mean via pandas."""
    return pd.Series(arr).rolling(window, center=True, min_periods=1).mean().values

def save(fig, name: str):
    fp = PLOTS / name
    fig.savefig(fp, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved → {fp}")

print("Setup complete. Output folder:", PLOTS)
print("Methods:", list(METHODS.keys()))



fig, axes = plt.subplots(3, 2, figsize=(14, 12))
fig.suptitle("Anomaly Score Distributions: Benign vs Attack\n(KDE estimate, 100 K sample)",
             fontsize=15, fontweight="bold", y=1.01)

for row_i, (label, mkey) in enumerate(METHODS.items()):
    for col_i, day in enumerate(DAYS):
        ax = axes[row_i][col_i]
        df = load_scores(mkey, day, sample=100_000)

        clip_val = float(np.percentile(df["score"].values, 99.5))
        benign   = df.loc[df["label"] == 0, "score"].values
        attacks  = df.loc[df["label"] == 1, "score"].values
        benign   = benign[benign <= clip_val]
        attacks  = attacks[attacks <= clip_val]

        x = np.linspace(0, clip_val, 400)
        for scores, color, name in [
            (benign,  LABEL_COLORS[0], "Benign"),
            (attacks, LABEL_COLORS[1], "Attack"),
        ]:
            if len(scores) > 30:
                # subsample for KDE speed
                if len(scores) > 20_000:
                    scores = np.random.default_rng(0).choice(scores, 20_000,
                                                             replace=False)
                kde_vals = gaussian_kde_numpy(scores, x)
                ax.plot(x, kde_vals, color=color, lw=2, label=name)
                ax.fill_between(x, kde_vals, alpha=0.15, color=color)

        ax.set_title(f"{label} — {DAY_CAP[day]}", fontsize=11, fontweight="bold")
        ax.set_xlabel("KitNET Anomaly Score")
        ax.set_ylabel("Density")
        ax.legend(fontsize=9)
        ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
save(fig, "score_distributions_kde.png")


# ════════════════════════════════════════════════════════════════════════════
# CELL 3 — Precision-Recall Curves (all 3 methods overlaid, per day)
# ════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Precision-Recall Curves — All Methods",
             fontsize=14, fontweight="bold")

for col_i, day in enumerate(DAYS):
    ax = axes[col_i]
    for label, mkey in METHODS.items():
        pr = load_curve("pr_curve.csv", mkey, day, step=1000)
        m  = load_metrics(mkey, day)
        baseline = m["attack_rate"]
        ax.plot(pr["recall"], pr["precision"],
                color=METHOD_COLORS[label], lw=2,
                label=f"{label}  (AUPRC={m['AUPRC']:.4f})")

    ax.axhline(baseline, color="gray", linestyle="--", lw=1.2,
               label=f"Random baseline ({baseline:.3f})")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("Recall", fontsize=11)
    ax.set_ylabel("Precision", fontsize=11)
    ax.set_title(DAY_CAP[day], fontsize=12, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
save(fig, "pr_curves_overlay.png")




fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Anomaly Score Distribution by Traffic Type\n"
             "(box=IQR, whiskers=1.5×IQR, outliers hidden, 50K sample)",
             fontsize=13, fontweight="bold", y=1.01)

ATYPES      = ["Benign", "Brute Force -Web", "SQL Injection"]
ATYPE_SHORT = {"Benign": "Benign", "Brute Force -Web": "BF-Web",
               "SQL Injection": "SQL-Inj"}
ATYPE_COLOR = {"Benign": "#2196F3", "Brute Force -Web": "#E53935",
               "SQL Injection": "#FF9800"}

for col_i, (label, mkey) in enumerate(METHODS.items()):
    for row_i, day in enumerate(DAYS):
        ax = axes[row_i][col_i]
        df = load_scores(mkey, day, sample=50_000)
        clip_val = float(np.percentile(df["score"].values, 99.5))
        df["score"] = df["score"].clip(upper=clip_val)

        groups, xticks, colors = [], [], []
        for at in ATYPES:
            grp = df.loc[df["attack_type"] == at, "score"].dropna().values
            if len(grp) > 0:
                groups.append(grp)
                xticks.append(ATYPE_SHORT[at])
                colors.append(ATYPE_COLOR[at])

        if groups:
            bp = ax.boxplot(groups, patch_artist=True, notch=False,
                            showfliers=False, widths=0.5)
            for patch, color in zip(bp["boxes"], colors):
                patch.set_facecolor(color); patch.set_alpha(0.6)

        ax.set_xticks(range(1, len(groups) + 1))
        ax.set_xticklabels(xticks, fontsize=9)
        ax.set_title(f"{label}\n{DAY_CAP[day]}", fontsize=10, fontweight="bold")
        ax.set_ylabel("Anomaly Score")
        ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
save(fig, "score_boxplots_by_attack_type.png")



WINDOW = 2000   # rows

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Anomaly Score Over Row Stream (temporal proxy)\n"
             f"Smoothed with window={WINDOW:,}",
             fontsize=13, fontweight="bold", y=1.01)

for col_i, (label, mkey) in enumerate(METHODS.items()):
    for row_i, day in enumerate(DAYS):
        ax = axes[row_i][col_i]
        path = RESULTS / mkey / day / "scores.csv"
        df = pd.read_csv(path).sort_values("row_index").reset_index(drop=True)
        clip_val = float(np.percentile(df["score"].values, 99.5))
        df["score"] = df["score"].clip(upper=clip_val)

        smooth = rolling_mean_np(df["score"].values, WINDOW)
        idx    = np.arange(len(df))

        # Shade attack regions
        attack_mask = (df["label"].values == 1)
        # group contiguous attack blocks for efficient shading
        in_block = False
        start_i  = 0
        for i, is_atk in enumerate(attack_mask):
            if is_atk and not in_block:
                start_i  = i; in_block = True
            elif not is_atk and in_block:
                ax.axvspan(start_i, i, color="#FFCDD2", alpha=0.3)
                in_block = False
        if in_block:
            ax.axvspan(start_i, len(attack_mask), color="#FFCDD2", alpha=0.3)

        ax.plot(idx, smooth, color=METHOD_COLORS[label], lw=1.2,
                label="Smoothed score")

        # dummy patch for legend
        from matplotlib.patches import Patch
        legend_elements = [
            plt.Line2D([0], [0], color=METHOD_COLORS[label], lw=2,
                       label="Smoothed score"),
            Patch(facecolor="#FFCDD2", alpha=0.5, label="Attack region"),
        ]
        ax.legend(handles=legend_elements, fontsize=8, loc="upper right")
        ax.set_title(f"{label} — {DAY_CAP[day]}", fontsize=10, fontweight="bold")
        ax.set_xlabel("Row index (stream order)")
        ax.set_ylabel("Anomaly Score")
        ax.grid(alpha=0.2)

plt.tight_layout()
save(fig, "temporal_score_plot.png")


# ════════════════════════════════════════════════════════════════════════════
# CELL 6 — Confusion Matrix Heatmaps
# ════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle("Confusion Matrices at Optimal Threshold",
             fontsize=14, fontweight="bold", y=1.02)

for col_i, (label, mkey) in enumerate(METHODS.items()):
    for row_i, day in enumerate(DAYS):
        ax = axes[row_i][col_i]
        m  = load_metrics(mkey, day)
        cm = np.array([[m["TN"], m["FP"]],
                       [m["FN"], m["TP"]]], dtype=np.int64)
        total  = cm.sum()
        cm_pct = cm / total * 100

        im = ax.imshow(cm_pct, cmap="Blues", vmin=0, vmax=100)

        for i in range(2):
            for j in range(2):
                color = "white" if cm_pct[i, j] > 50 else "black"
                ax.text(j, i,
                        f"{cm[i, j]:,}\n({cm_pct[i, j]:.1f}%)",
                        ha="center", va="center",
                        fontsize=9, color=color, fontweight="bold")

        ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
        ax.set_xticklabels(["Pred Benign", "Pred Attack"])
        ax.set_yticklabels(["True Benign", "True Attack"])
        ax.set_title(
            f"{label} — {DAY_CAP[day]}\n"
            f"F1={m['F1_optimal']:.4f}  thr={m['threshold_opt']:.4f}",
            fontsize=9, fontweight="bold"
        )
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04).set_label("% of total")

plt.tight_layout()
save(fig, "confusion_matrices.png")


# ════════════════════════════════════════════════════════════════════════════
# CELL 7 — Score CDF: Benign vs Attack
# ════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Cumulative Distribution of Anomaly Scores\n(Benign vs Attack, 100K sample)",
             fontsize=13, fontweight="bold", y=1.01)

for col_i, (label, mkey) in enumerate(METHODS.items()):
    for row_i, day in enumerate(DAYS):
        ax = axes[row_i][col_i]
        df = load_scores(mkey, day, sample=100_000)
        clip_val = float(np.percentile(df["score"].values, 99.5))

        for lbl, color, name in [(0, LABEL_COLORS[0], "Benign"),
                                  (1, LABEL_COLORS[1], "Attack")]:
            vals = df.loc[df["label"] == lbl, "score"].values
            vals = np.sort(vals[vals <= clip_val])
            cdf  = np.arange(1, len(vals) + 1) / len(vals)
            ax.plot(vals, cdf, color=color, lw=2, label=name)

        # 95th percentile of benign = detection threshold
        benign_scores = df.loc[df["label"] == 0, "score"].values
        thr95 = float(np.percentile(benign_scores, 95))
        ax.axvline(thr95, color="orange", linestyle="--", lw=1.5,
                   label=f"p95 thr={thr95:.3f}")

        ax.set_title(f"{label} — {DAY_CAP[day]}", fontsize=10, fontweight="bold")
        ax.set_xlabel("Anomaly Score")
        ax.set_ylabel("Cumulative Probability")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
        ax.set_xlim(left=0, right=clip_val)

plt.tight_layout()
save(fig, "score_cdf.png")


# ════════════════════════════════════════════════════════════════════════════
# CELL 8 — Metrics Summary Heatmap
# ════════════════════════════════════════════════════════════════════════════

rows = []
for label, mkey in METHODS.items():
    abd = load_attack_breakdown(mkey)
    for day in DAYS:
        d   = abd[abd["day"] == day]
        bf  = d[d["attack_type"] == "Brute Force -Web"]["detection_rate"].values
        sql = d[d["attack_type"] == "SQL Injection"]["detection_rate"].values
        m   = load_metrics(mkey, day)
        rows.append({
            "Method":      label,
            "Day":         DAY_CAP[day],
            "AUC":         round(m["AUC"],        4),
            "EER":         round(m["EER"],        4),
            "F1":          round(m["F1_optimal"], 4),
            "BF-Web Det%": round(float(bf[0])  * 100, 1) if len(bf)  else 0.0,
            "SQL Det%":    round(float(sql[0]) * 100, 1) if len(sql) else 0.0,
        })

sumdf = pd.DataFrame(rows)
sumdf["Exp"] = sumdf["Method"] + "\n" + sumdf["Day"]
sumdf = sumdf.set_index("Exp")

METRICS = ["AUC", "EER", "F1", "BF-Web Det%", "SQL Det%"]
heat_data = sumdf[METRICS]
mat = heat_data.values.astype(float)

# Normalise column-wise (0 → worst, 1 → best)
# EER: lower = better, so invert
mat_norm = np.zeros_like(mat)
for ci, metric in enumerate(METRICS):
    col = mat[:, ci]
    rng = col.max() - col.min()
    if rng < 1e-9:
        mat_norm[:, ci] = 0.5
    elif metric == "EER":
        mat_norm[:, ci] = 1 - (col - col.min()) / rng   # invert
    else:
        mat_norm[:, ci] = (col - col.min()) / rng

fig, ax = plt.subplots(figsize=(12, 7))
fig.suptitle("Metrics Summary Heatmap", fontsize=14, fontweight="bold")

im = ax.imshow(mat_norm.T, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)

ax.set_xticks(range(len(heat_data.index)))
ax.set_xticklabels(heat_data.index.tolist(), fontsize=9)
ax.set_yticks(range(len(METRICS)))
ax.set_yticklabels(METRICS, fontsize=11)

for i, metric in enumerate(METRICS):
    for j in range(len(heat_data.index)):
        val = heat_data.iloc[j][metric]
        nrm = mat_norm[j, i]
        color = "white" if nrm < 0.2 or nrm > 0.8 else "black"
        ax.text(j, i, str(val), ha="center", va="center",
                fontsize=9, fontweight="bold", color=color)

plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02).set_label(
    "Normalised (0=worst, 1=best per metric; EER inverted)")
ax.set_title("Green = better, Red = worse  (column-normalised)",
             fontsize=9, style="italic")
plt.tight_layout()
save(fig, "metrics_summary_heatmap.png")


# ════════════════════════════════════════════════════════════════════════════
# CELL 9 — Score-Gap Bar Chart (mean_attack − mean_benign)
# ════════════════════════════════════════════════════════════════════════════

gap_rows = []
for label, mkey in METHODS.items():
    for day in DAYS:
        m = load_metrics(mkey, day)
        gap = m["mean_score_attack"] - m["mean_score_benign"]
        gap_rows.append({
            "Method":      label,
            "Day":         DAY_CAP[day],
            "Gap":         round(gap, 6),
            "mean_benign": round(m["mean_score_benign"], 4),
            "mean_attack": round(m["mean_score_attack"], 4),
        })

gdf = pd.DataFrame(gap_rows)

fig, ax = plt.subplots(figsize=(10, 6))
mnames = list(METHODS.keys())
x = np.arange(len(mnames))
width = 0.35
day_colors = ["#5C6BC0", "#EF5350"]

for i, day in enumerate(DAYS):
    vals = [
        gdf[(gdf["Method"] == lbl) & (gdf["Day"] == DAY_CAP[day])]["Gap"].values[0]
        for lbl in mnames
    ]
    bars = ax.bar(x + i * width - width / 2, vals, width,
                  label=DAY_CAP[day], color=day_colors[i],
                  alpha=0.85, edgecolor="white", linewidth=0.8)
    for bar, val in zip(bars, vals):
        va   = "bottom" if val >= 0 else "top"
        ypos = val + 0.0005 if val >= 0 else val - 0.0005
        ax.text(bar.get_x() + bar.get_width() / 2, ypos,
                f"{val:+.4f}", ha="center", va=va,
                fontsize=8, fontweight="bold")

ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(mnames, fontsize=10)
ax.set_ylabel("Score Gap  (mean_attack − mean_benign)", fontsize=11)
ax.set_title("Score Separation: Attack vs Benign Mean Score\n"
             "Positive = attacks score higher (correct);  "
             "Negative = inverted signal (Traditional/Hybrid only)",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
save(fig, "score_gap_analysis.png")


# ════════════════════════════════════════════════════════════════════════════
# ALL DONE
# ════════════════════════════════════════════════════════════════════════════

print("\nAll graphs saved to:", PLOTS)
for f in sorted(PLOTS.glob("*.png")):
    print(" ", f.name)


Setup complete. Output folder: /kaggle/working/extra_graphs
Methods: ['Traditional 115', 'HTTP-only 85', 'Hybrid 200']
Saved → /kaggle/working/extra_graphs/score_distributions_kde.png
Saved → /kaggle/working/extra_graphs/pr_curves_overlay.png
Saved → /kaggle/working/extra_graphs/score_boxplots_by_attack_type.png
Saved → /kaggle/working/extra_graphs/temporal_score_plot.png
Saved → /kaggle/working/extra_graphs/confusion_matrices.png
Saved → /kaggle/working/extra_graphs/score_cdf.png
Saved → /kaggle/working/extra_graphs/metrics_summary_heatmap.png
Saved → /kaggle/working/extra_graphs/score_gap_analysis.png

All graphs saved to: /kaggle/working/extra_graphs
  confusion_matrices.png
  metrics_summary_heatmap.png
  pr_curves_overlay.png
  score_boxplots_by_attack_type.png
  score_cdf.png
  score_distributions_kde.png
  score_gap_analysis.png
  temporal_score_plot.png


In [10]:
!zip -r my_folders.zip /kaggle/working/results_phase2 /kaggle/working/extra_graphs

  adding: kaggle/working/results_phase2/ (stored 0%)
  adding: kaggle/working/results_phase2/full/ (stored 0%)
  adding: kaggle/working/results_phase2/full/traditional/ (stored 0%)
  adding: kaggle/working/results_phase2/full/traditional/summary_metrics.csv (deflated 52%)
  adding: kaggle/working/results_phase2/full/traditional/_plots/ (stored 0%)
  adding: kaggle/working/results_phase2/full/traditional/_plots/summary_runtime.png (deflated 30%)
  adding: kaggle/working/results_phase2/full/traditional/_plots/thursday_score_dist.png (deflated 24%)
  adding: kaggle/working/results_phase2/full/traditional/_plots/combined_roc.png (deflated 12%)
  adding: kaggle/working/results_phase2/full/traditional/_plots/summary_auc.png (deflated 31%)
  adding: kaggle/working/results_phase2/full/traditional/_plots/summary_metrics_sorted.csv (deflated 52%)
  adding: kaggle/working/results_phase2/full/traditional/_plots/friday_score_dist.png (deflated 24%)
  adding: kaggle/working/results_phase2/full/tradi